# NILM — Joint Multi-Appliance Model (CNN + BiLSTM + Knowledge Fusion)

Single shared model across all 5 appliances (`laptop`, `washing_machine`, `tv`, `fridge`,
`microwave`) on the 2-year UK-DALE House 1 CSV, matching this architecture:

```
Aggregate power -> Sliding window (100 samples)
    -> 1D CNN -> BiLSTM -> Temporal attention           (learned feature)
    -> Knowledge feature extractor (12 features)         (knowledge vector)
    -> Knowledge fusion layer (gate + concat)
    -> Appliance state (multi-label) / Duty cycle / Knowledge reconstruction
    -> Graph reasoner (min-dwell correction)
    -> Activity timeline + Explainability report
```

**Design choices made while building this** (see the note after each section for why):
- The 12 knowledge features are power-derived only — no time-of-day/day-of-week. An earlier
  version of this architecture (single-appliance) let the model substitute "usual hour of use"
  for actually reading the power trace; excluding temporal features here removes that failure
  mode by construction rather than needing to catch it after the fact.
- Per-appliance ON thresholds are picked automatically from the data (2-cluster split on
  log-power), not guessed.
- `pos_weight` is capped per appliance — an uncapped ratio previously taught a model to guess
  "ON" too often for rare appliances.
- Decision thresholds are swept per appliance to maximize F-beta (not a blanket 0.5), with a
  higher beta (recall-weighted) for microwave.
- The graph reasoner's minimum dwell time comes from each appliance's *real* median ON duration
  in the training data, not a guessed constant.

**One real limitation of this architecture, flagged rather than hidden:** because there is a
single shared CNN backbone reading one shared window for all 5 appliances, the window size can't
be tuned per appliance the way it could in a specialist-per-appliance design. Microwave's bursts
(2-5 min) inside a 10-minute (100-sample) window are likely to be diluted the same way they were
before that fix was applied elsewhere. Section 6 below measures this directly so it's visible
rather than discovered later. If microwave's numbers come out weak here, that's the expected
tradeoff of one-model-for-everything, not a bug in this notebook.


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

import os, math, json as _json, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, f1_score, fbeta_score, precision_score, recall_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Mounted at /content/gdrive
device: cpu


## 1. Config

In [ ]:
CSV_PATH = "/content/gdrive/MyDrive/house_1/house_1_fully/house1_required_2years.csv"  # same file used in the Approach2 stacking notebook

SAVE_DIR = "/content/gdrive/MyDrive/NILM_JointModel"
os.makedirs(SAVE_DIR, exist_ok=True)

APPLIANCES = ["laptop", "washing_machine", "tv", "fridge", "microwave"]
N_APPLIANCES = len(APPLIANCES)

WINDOW_SIZE = 100   # shared across all appliances -- see the architecture note above
STRIDE = 20

KV_DIM = 12          # matches the diagram's "12 engineered features"
BATCH_SIZE = 128

POS_WEIGHT_CAP = {   # per-appliance cap -- microwave's imbalance is usually the most severe of the five
    "laptop": 10.0, "washing_machine": 10.0, "tv": 10.0, "fridge": 10.0, "microwave": 15.0,
}

# recall matters more than precision for microwave specifically (missing real usage is worse
# than an occasional false alarm) -- everything else optimizes plain F1 (beta=1)
THRESHOLD_BETA = {
    "laptop": 1.0, "washing_machine": 1.0, "tv": 1.0, "fridge": 1.0, "microwave": 2.0,
}


## 2. Load + clean data

In [ ]:
df = pd.read_csv(CSV_PATH)

required_cols = ["datetime", "aggregate"] + APPLIANCES
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV is missing expected columns: {missing}. Update required_cols/CSV_PATH.")

df = df[required_cols].copy()
for c in ["aggregate"] + APPLIANCES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["aggregate"] + APPLIANCES).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
df["time_s"] = (df["datetime"] - df["datetime"].iloc[0]).dt.total_seconds()

SAMPLE_INTERVAL_S = float(df["datetime"].diff().dt.total_seconds().median())

n = len(df)
TRAIN_CUTOFF = int(n * 0.70)
VAL_CUTOFF = int(n * 0.85)

print(f"rows: {n:,}  sample_interval: {SAMPLE_INTERVAL_S:.1f}s")
print(f"train: 0-{TRAIN_CUTOFF:,}  val: {TRAIN_CUTOFF:,}-{VAL_CUTOFF:,}  test: {VAL_CUTOFF:,}-{n:,}")


rows: 9,469,092  sample_interval: 6.0s
train: 0-6,628,364  val: 6,628,364-8,048,728  test: 8,048,728-9,469,092


## 3. Per-appliance ON threshold (auto-picked, not guessed)

2-cluster split on each appliance's own non-zero TRAIN-period power values (log scale), threshold
= geometric mean of the two cluster centers. A single blanket threshold across all five would be
wrong for at least one of them (a microwave draws far more power when on than a laptop does).


In [ ]:
def auto_pick_threshold(power_values, min_threshold=3.0):
    vals = power_values[power_values > 0]
    if len(vals) < 50:
        return min_threshold
    log_vals = np.log1p(vals).reshape(-1, 1)
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(log_vals)
    centers = np.expm1(km.cluster_centers_.ravel())
    threshold = float(np.sqrt(max(centers.min(), 1e-6) * max(centers.max(), 1e-6)))
    return max(min_threshold, threshold)


ON_THRESHOLD = {}
print(f"{'appliance':16s} {'threshold(W)':>12s} {'mean':>8s} {'p50':>8s} {'p90':>8s} {'max':>8s}")
for app in APPLIANCES:
    train_power = df[app].values[:TRAIN_CUTOFF]
    thr = auto_pick_threshold(train_power)
    ON_THRESHOLD[app] = thr
    print(f"{app:16s} {thr:12.1f} {train_power.mean():8.1f} {np.percentile(train_power,50):8.1f} "
          f"{np.percentile(train_power,90):8.1f} {train_power.max():8.1f}")

print()
print("Sanity-check these against what you know about each appliance before trusting them.")


appliance        threshold(W)     mean      p50      p90      max
laptop                    4.6      4.0      0.0     19.0    544.0
washing_machine          57.0     23.6      0.0      0.0   3999.0
tv                       10.2     10.2      1.0      1.0   2049.0
fridge                  148.4     35.1      0.0     90.0   3323.0
microwave                28.7      4.5      1.0      1.0   3206.0

Sanity-check these against what you know about each appliance before trusting them.


## 4. Multi-label ON/OFF states

In [ ]:
Y_state_full = np.zeros((len(df), N_APPLIANCES), dtype=np.float32)
for j, app in enumerate(APPLIANCES):
    Y_state_full[:, j] = (df[app].values >= ON_THRESHOLD[app]).astype(np.float32)

print(f"{'appliance':16s} {'ON %':>8s}")
for j, app in enumerate(APPLIANCES):
    print(f"{app:16s} {100*Y_state_full[:, j].mean():8.2f}")


appliance            ON %
laptop              18.17
washing_machine      3.61
tv                   9.06
fridge               1.23
microwave            0.42


## 5. Capped `pos_weight` per appliance

Same fix applied throughout this project: an uncapped `neg/pos` ratio in BCE loss doesn't just
balance classes, it rewards guessing ON almost regardless of evidence once the ratio gets large.


In [ ]:
raw_pos_weight = np.zeros(N_APPLIANCES, dtype=np.float32)
capped_pos_weight = np.zeros(N_APPLIANCES, dtype=np.float32)

train_state = Y_state_full[:TRAIN_CUTOFF]
print(f"{'appliance':16s} {'pos_frac':>9s} {'raw':>8s} {'capped':>7s}")
for j, app in enumerate(APPLIANCES):
    pos_frac = train_state[:, j].mean()
    raw = (1 - pos_frac) / max(pos_frac, 1e-6)
    cap = POS_WEIGHT_CAP[app]
    capped = min(raw, cap)
    raw_pos_weight[j] = raw
    capped_pos_weight[j] = capped
    flag = "  <-- capped" if raw > cap else ""
    print(f"{app:16s} {pos_frac:9.4f} {raw:8.1f} {capped:7.1f}{flag}")

pos_weight_tensor = torch.tensor(capped_pos_weight, dtype=torch.float32).to(device)


appliance         pos_frac      raw  capped
laptop              0.1994      4.0     4.0
washing_machine     0.0345     28.0    10.0  <-- capped
tv                  0.0894     10.2    10.0  <-- capped
fridge              0.0120     82.1    10.0  <-- capped
microwave           0.0029    341.4    15.0  <-- capped


## 6. Event-duration diagnostic -- is the shared 100-sample window a good fit for each appliance?

See the architecture note at the top. This makes the tradeoff visible rather than discovering it
after training.


In [ ]:
def event_runs(binary):
    binary = binary.astype(int)
    if binary.sum() == 0 or binary.sum() == len(binary):
        return np.array([]), np.array([])
    changes = np.diff(binary)
    on_starts = np.where(changes == 1)[0] + 1
    on_ends = np.where(changes == -1)[0] + 1
    if binary[0] == 1:
        on_starts = np.insert(on_starts, 0, 0)
    if binary[-1] == 1:
        on_ends = np.append(on_ends, len(binary))
    n = min(len(on_starts), len(on_ends))
    on_starts, on_ends = on_starts[:n], on_ends[:n]
    return on_ends - on_starts, (on_starts[1:] - on_ends[:-1] if n > 1 else np.array([]))


window_span_s = WINDOW_SIZE * SAMPLE_INTERVAL_S
print(f"shared window span = {window_span_s:.0f}s (~{window_span_s/60:.1f} min)\n")
print(f"{'appliance':16s} {'median_on(s)':>13s} {'p90_on(s)':>10s}")
for j, app in enumerate(APPLIANCES):
    on_durations, _ = event_runs(train_state[:, j].astype(int))
    if len(on_durations) == 0:
        print(f"{app:16s}  no ON events found")
        continue
    med_s = np.median(on_durations) * SAMPLE_INTERVAL_S
    p90_s = np.percentile(on_durations, 90) * SAMPLE_INTERVAL_S
    flag = "  <-- shorter than window: likely diluted, expect weaker recall for this one" if med_s < window_span_s * 0.5 else ""
    print(f"{app:16s} {med_s:13.0f} {p90_s:10.0f}{flag}")


shared window span = 600s (~10.0 min)

appliance         median_on(s)  p90_on(s)
laptop                     204       2406  <-- shorter than window: likely diluted, expect weaker recall for this one
washing_machine             12         60  <-- shorter than window: likely diluted, expect weaker recall for this one
tv                          54        156  <-- shorter than window: likely diluted, expect weaker recall for this one
fridge                      12        120  <-- shorter than window: likely diluted, expect weaker recall for this one
microwave                   30         72  <-- shorter than window: likely diluted, expect weaker recall for this one


## 7. Knowledge vector -- 12 generic, power-only features (no temporal)

Deliberately excludes hour/day-of-week encodings, per the note at the top of this notebook.


In [ ]:
KV_FEATURE_NAMES = [
    "mean_power", "max_power", "min_power", "std_power", "power_range",
    "duty_cycle", "num_transitions", "largest_jump",
    "rise_time_frac", "fall_time_frac", "spectral_entropy", "skewness",
]
assert len(KV_FEATURE_NAMES) == KV_DIM


def compute_knowledge_vector(window, generic_on_threshold=15.0):
    mean_p = window.mean(); max_p = window.max(); min_p = window.min(); std_p = window.std()
    power_range = max_p - min_p
    binary = (window > generic_on_threshold).astype(float)
    duty_cycle = binary.mean()
    num_transitions = float(np.sum(np.abs(np.diff(binary))))
    diffs = np.diff(window)
    largest_jump = float(np.max(np.abs(diffs))) if len(diffs) else 0.0

    peak_idx = int(np.argmax(window))
    rise_time_frac = peak_idx / max(len(window) - 1, 1)
    fall_time_frac = (len(window) - 1 - peak_idx) / max(len(window) - 1, 1)

    fft_mag = np.abs(np.fft.rfft(window - mean_p)) + 1e-12
    p = fft_mag / fft_mag.sum()
    spectral_entropy = float(-np.sum(p * np.log(p)) / np.log(len(p)))

    skewness = float(((window - mean_p) ** 3).mean() / (std_p ** 3 + 1e-6))

    return np.array([
        mean_p, max_p, min_p, std_p, power_range,
        duty_cycle, num_transitions, largest_jump,
        rise_time_frac, fall_time_frac, spectral_entropy, skewness,
    ], dtype=np.float32)


## 8. Dataset -- sliding windows, multi-label center-point state + duty-cycle targets

In [ ]:
class JointNILMDataset(Dataset):
    def __init__(self, df, y_state, start_idx, end_idx, kv_mean=None, kv_std=None,
                 window_size=WINDOW_SIZE, stride=STRIDE):
        agg = df["aggregate"].values.astype(np.float32)
        starts = list(range(start_idx, end_idx - window_size, stride))

        windows, kvs, labels, duty = [], [], [], []
        for s in starts:
            e = s + window_size
            w = agg[s:e]
            center_idx = s + window_size // 2
            label = y_state[center_idx]                       # (N_APPLIANCES,)
            duty_frac = y_state[s:e].mean(axis=0)              # fraction ON per appliance in window
            kv = compute_knowledge_vector(w)
            windows.append((w - w.mean()) / (w.std() + 1e-6))
            kvs.append(kv)
            labels.append(label)
            duty.append(duty_frac)

        self.agg = np.stack(windows).astype(np.float32)
        self.kv_raw = np.stack(kvs).astype(np.float32)
        self.labels = np.stack(labels).astype(np.float32)
        self.duty = np.stack(duty).astype(np.float32)

        if kv_mean is None:
            self.kv_mean = self.kv_raw.mean(axis=0)
            self.kv_std = self.kv_raw.std(axis=0) + 1e-6
        else:
            self.kv_mean, self.kv_std = kv_mean, kv_std
        self.kv = (self.kv_raw - self.kv_mean) / self.kv_std

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "aggregate": torch.tensor(self.agg[idx]).unsqueeze(0),
            "kv": torch.tensor(self.kv[idx]),
            "labels": torch.tensor(self.labels[idx]),
            "duty": torch.tensor(self.duty[idx]),
        }


train_ds = JointNILMDataset(df, Y_state_full, 0, TRAIN_CUTOFF)
val_ds = JointNILMDataset(df, Y_state_full, TRAIN_CUTOFF, VAL_CUTOFF, kv_mean=train_ds.kv_mean, kv_std=train_ds.kv_std)
test_ds = JointNILMDataset(df, Y_state_full, VAL_CUTOFF, len(df), kv_mean=train_ds.kv_mean, kv_std=train_ds.kv_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

print(f"train windows: {len(train_ds):,}  val: {len(val_ds):,}  test: {len(test_ds):,}")


train windows: 331,414  val: 71,014  test: 71,014


## 9. Model -- CNN + BiLSTM + temporal attention, fused with knowledge branch, 3 heads

In [ ]:
class CNNBranch(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=7, stride=2, padding=3), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2), nn.ReLU(),
        )
    def forward(self, x):
        return self.net(x)


class TemporalAttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)
    def forward(self, seq):
        scores = self.score(seq).squeeze(-1)
        weights = F.softmax(scores, dim=-1)
        context = torch.einsum("bt,btd->bd", weights, seq)
        return context, weights


class KnowledgeEncoder(nn.Module):
    def __init__(self, kv_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(kv_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim), nn.ReLU())
    def forward(self, kv):
        return self.net(kv)


class NILMKnowledgeNet(nn.Module):
    def __init__(self, kv_dim=KV_DIM, n_appliances=N_APPLIANCES, lstm_hidden=64, knowledge_embed_dim=32):
        super().__init__()
        self.cnn = CNNBranch()
        self.bilstm = nn.LSTM(64, lstm_hidden, batch_first=True, bidirectional=True)
        context_dim = lstm_hidden * 2
        self.temporal_attn = TemporalAttentionPool(context_dim)

        self.knowledge_encoder = KnowledgeEncoder(kv_dim, knowledge_embed_dim)
        self.gate = nn.Sequential(nn.Linear(knowledge_embed_dim, context_dim), nn.Sigmoid())

        fusion_dim = context_dim + knowledge_embed_dim
        self.state_head = nn.Linear(fusion_dim, n_appliances)
        self.duty_head = nn.Sequential(nn.Linear(fusion_dim, n_appliances), nn.Sigmoid())
        self.knowledge_head = nn.Linear(fusion_dim, kv_dim)

    def forward(self, aggregate, kv):
        cnn_feat = self.cnn(aggregate)
        cnn_seq = cnn_feat.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(cnn_seq)
        context, attn_weights = self.temporal_attn(lstm_out)

        kv_embed = self.knowledge_encoder(kv)
        gated_context = context * self.gate(kv_embed)
        fusion = torch.cat([gated_context, kv_embed], dim=-1)

        return {
            "state_logits": self.state_head(fusion),
            "duty_pred": self.duty_head(fusion),
            "kv_pred": self.knowledge_head(fusion),
            "attn_weights": attn_weights,
        }


## 10. Light hyperparameter search

In [ ]:
import random

def evaluate_macro_f1(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch["aggregate"].to(device), batch["kv"].to(device))
            all_preds.append((torch.sigmoid(out["state_logits"]) > 0.5).cpu().numpy())
            all_labels.append(batch["labels"].numpy())
    preds = np.concatenate(all_preds); labels_np = np.concatenate(all_labels)
    f1s = [f1_score(labels_np[:, i], preds[:, i], zero_division=0) for i in range(labels_np.shape[1])]
    return f1s


def build_and_train(lr, lstm_hidden, knowledge_embed_dim, knowledge_loss_weight, duty_loss_weight, n_epochs=4):
    m = NILMKnowledgeNet(lstm_hidden=lstm_hidden, knowledge_embed_dim=knowledge_embed_dim).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    best_f1 = 0.0
    for epoch in range(n_epochs):
        m.train()
        for batch in train_loader:
            agg = batch["aggregate"].to(device); kv = batch["kv"].to(device)
            labels = batch["labels"].to(device); duty = batch["duty"].to(device)
            out = m(agg, kv)
            loss = (F.binary_cross_entropy_with_logits(out["state_logits"], labels, pos_weight=pos_weight_tensor)
                    + knowledge_loss_weight * F.mse_loss(out["kv_pred"], kv)
                    + duty_loss_weight * F.mse_loss(out["duty_pred"], duty))
            opt.zero_grad(); loss.backward(); opt.step()
        macro_f1 = np.mean(evaluate_macro_f1(m, val_loader))
        best_f1 = max(best_f1, macro_f1)
    return best_f1


search_space = {
    "lr": [1e-3, 5e-4, 2e-3],
    "lstm_hidden": [64, 96],
    "knowledge_embed_dim": [32, 48],
    "knowledge_loss_weight": [0.3, 0.5, 1.0],
    "duty_loss_weight": [0.5, 1.0],
}

N_TRIALS = 10
results = []
for trial in range(N_TRIALS):
    config = {k: random.choice(v) for k, v in search_space.items()}
    f1 = build_and_train(**config)
    print(f"trial {trial+1:2d}: {config} -> macro F1 {f1:.3f}")
    results.append((f1, config))

results.sort(key=lambda x: -x[0])
best_config = results[0][1]
print("\nBest config:", results[0])


trial  1: {'lr': 0.002, 'lstm_hidden': 64, 'knowledge_embed_dim': 48, 'knowledge_loss_weight': 0.3, 'duty_loss_weight': 1.0} -> macro F1 0.620
trial  2: {'lr': 0.0005, 'lstm_hidden': 64, 'knowledge_embed_dim': 32, 'knowledge_loss_weight': 0.5, 'duty_loss_weight': 0.5} -> macro F1 0.585
trial  3: {'lr': 0.001, 'lstm_hidden': 64, 'knowledge_embed_dim': 48, 'knowledge_loss_weight': 0.5, 'duty_loss_weight': 1.0} -> macro F1 0.620
trial  4: {'lr': 0.002, 'lstm_hidden': 96, 'knowledge_embed_dim': 48, 'knowledge_loss_weight': 0.5, 'duty_loss_weight': 0.5} -> macro F1 0.622
trial  5: {'lr': 0.0005, 'lstm_hidden': 96, 'knowledge_embed_dim': 32, 'knowledge_loss_weight': 0.3, 'duty_loss_weight': 0.5} -> macro F1 0.597
trial  6: {'lr': 0.001, 'lstm_hidden': 64, 'knowledge_embed_dim': 32, 'knowledge_loss_weight': 1.0, 'duty_loss_weight': 1.0} -> macro F1 0.613
trial  7: {'lr': 0.001, 'lstm_hidden': 64, 'knowledge_embed_dim': 48, 'knowledge_loss_weight': 0.5, 'duty_loss_weight': 1.0} -> macro F1 0.6

## 11. Full training run

In [ ]:
model = NILMKnowledgeNet(
    lstm_hidden=best_config["lstm_hidden"],
    knowledge_embed_dim=best_config["knowledge_embed_dim"],
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=best_config["lr"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

KNOWLEDGE_LOSS_WEIGHT = best_config["knowledge_loss_weight"]
DUTY_LOSS_WEIGHT = best_config["duty_loss_weight"]

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_state, total_kv, total_duty, n_batches = 0.0, 0.0, 0.0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        agg = batch["aggregate"].to(device); kv = batch["kv"].to(device)
        labels = batch["labels"].to(device); duty = batch["duty"].to(device)
        with torch.set_grad_enabled(is_train):
            out = model(agg, kv)
            state_loss = F.binary_cross_entropy_with_logits(out["state_logits"], labels, pos_weight=pos_weight_tensor)
            kv_loss = F.mse_loss(out["kv_pred"], kv)
            duty_loss = F.mse_loss(out["duty_pred"], duty)
            loss = state_loss + KNOWLEDGE_LOSS_WEIGHT * kv_loss + DUTY_LOSS_WEIGHT * duty_loss
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_state += state_loss.item(); total_kv += kv_loss.item(); total_duty += duty_loss.item(); n_batches += 1
        all_preds.append(torch.sigmoid(out["state_logits"]).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    preds = np.concatenate(all_preds) > 0.5
    labels_np = np.concatenate(all_labels)
    f1s = [f1_score(labels_np[:, i], preds[:, i], zero_division=0) for i in range(labels_np.shape[1])]
    return total_state/n_batches, total_kv/n_batches, total_duty/n_batches, f1s


N_EPOCHS = 30
best_macro_f1 = 0.0
CHECKPOINT_PATH = os.path.join(SAVE_DIR, "joint_model_best.pt")

for epoch in range(1, N_EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer)
    val = run_epoch(model, val_loader)
    scheduler.step(val[0])

    macro_f1 = np.mean(val[3])
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    val_f1_str = " ".join(f"{a[:4]}={f:.2f}" for a, f in zip(APPLIANCES, val[3]))
    print(f"epoch {epoch:2d} | train_loss {tr[0]:.3f} | val_loss {val[0]:.3f} F1[{val_f1_str}] "
          f"(best macro F1: {best_macro_f1:.3f})")

model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()
print(f"\nLoaded best checkpoint, macro F1 = {best_macro_f1:.3f}")


epoch  1 | train_loss 0.348 | val_loss 0.278 F1[lapt=0.54 wash=0.76 tv=0.69 frid=0.61 micr=0.16] (best macro F1: 0.552)
epoch  2 | train_loss 0.310 | val_loss 0.274 F1[lapt=0.53 wash=0.79 tv=0.73 frid=0.66 micr=0.33] (best macro F1: 0.609)
epoch  3 | train_loss 0.299 | val_loss 0.248 F1[lapt=0.55 wash=0.79 tv=0.69 frid=0.65 micr=0.35] (best macro F1: 0.609)
epoch  4 | train_loss 0.292 | val_loss 0.258 F1[lapt=0.54 wash=0.80 tv=0.69 frid=0.69 micr=0.31] (best macro F1: 0.609)
epoch  5 | train_loss 0.287 | val_loss 0.258 F1[lapt=0.55 wash=0.79 tv=0.72 frid=0.71 micr=0.36] (best macro F1: 0.629)
epoch  6 | train_loss 0.282 | val_loss 0.256 F1[lapt=0.53 wash=0.80 tv=0.71 frid=0.73 micr=0.36] (best macro F1: 0.629)
epoch  7 | train_loss 0.278 | val_loss 0.245 F1[lapt=0.54 wash=0.80 tv=0.71 frid=0.72 micr=0.33] (best macro F1: 0.629)
epoch  8 | train_loss 0.275 | val_loss 0.250 F1[lapt=0.54 wash=0.80 tv=0.72 frid=0.67 micr=0.33] (best macro F1: 0.629)
epoch  9 | train_loss 0.271 | val_loss 0

## 12. Shortcut diagnostic -- is the knowledge branch contributing real signal?

No temporal features are in the knowledge vector by design, but worth confirming the signal
branch (CNN+BiLSTM) is doing real work rather than the model leaning entirely on the 12
knowledge features.


In [ ]:
@torch.no_grad()
def shortcut_check(model, loader):
    model.eval()
    shifts, attn_stds = [], []
    for batch in loader:
        agg = batch["aggregate"].to(device); kv = batch["kv"].to(device)
        out_full = model(agg, kv)
        out_ablated = model(agg, torch.zeros_like(kv))
        shift = (torch.sigmoid(out_full["state_logits"]) - torch.sigmoid(out_ablated["state_logits"])).abs()
        shifts.append(shift.cpu().numpy())
        attn_stds.append(out_full["attn_weights"].std(dim=1).cpu().numpy())
    return np.concatenate(shifts), np.concatenate(attn_stds)


shifts, attn_stds = shortcut_check(model, val_loader)
print(f"{'appliance':16s} {'mean |shift| (kv->0)':>22s}")
for j, app in enumerate(APPLIANCES):
    print(f"{app:16s} {shifts[:, j].mean():22.3f}")
print(f"\nmean attention std across all windows: {attn_stds.mean():.4f}")
print("Low attn_std would mean the BiLSTM attention is flat/uninformative -- same red flag pattern seen earlier in this project.")


appliance          mean |shift| (kv->0)
laptop                            0.315
washing_machine                   0.127
tv                                0.247
fridge                            0.027
microwave                         0.007

mean attention std across all windows: 0.0963
Low attn_std would mean the BiLSTM attention is flat/uninformative -- same red flag pattern seen earlier in this project.


## 13. Per-appliance threshold tuning (F-beta, on VAL set)

In [ ]:
def sweep_best_threshold(y_true, y_prob, beta, thresholds=np.linspace(0.05, 0.95, 19)):
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        score = fbeta_score(y_true, pred, beta=beta, zero_division=0)
        if score > best_score:
            best_t, best_score = t, score
    return best_t, best_score


@torch.no_grad()
def get_val_probs(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        out = model(batch["aggregate"].to(device), batch["kv"].to(device))
        probs.append(torch.sigmoid(out["state_logits"]).cpu().numpy())
    return np.concatenate(probs)


VAL_PROBS = get_val_probs(model, val_loader)
val_labels = val_ds.labels

BEST_THRESHOLDS = {}
print(f"{'appliance':16s} {'beta':>5s} {'best_t':>7s} {'score@best':>11s} {'F1@0.5':>8s}")
for j, app in enumerate(APPLIANCES):
    beta = THRESHOLD_BETA[app]
    best_t, best_score = sweep_best_threshold(val_labels[:, j], VAL_PROBS[:, j], beta)
    f1_at_half = f1_score(val_labels[:, j], (VAL_PROBS[:, j] >= 0.5).astype(int), zero_division=0)
    BEST_THRESHOLDS[app] = float(best_t)
    print(f"{app:16s} {beta:5.1f} {best_t:7.2f} {best_score:11.3f} {f1_at_half:8.3f}")


appliance         beta  best_t  score@best   F1@0.5
laptop             1.0    0.70       0.604    0.529
washing_machine    1.0    0.85       0.828    0.810
tv                 1.0    0.80       0.757    0.729
fridge             1.0    0.60       0.708    0.696
microwave          2.0    0.20       0.507    0.443


## 14. Graph reasoner -- minimum dwell time from real per-appliance ON duration

In [ ]:
min_dwell_windows = {}
print(f"{'appliance':16s} {'median_on(s)':>13s} {'min_dwell(windows)':>19s}")
for j, app in enumerate(APPLIANCES):
    on_durations, _ = event_runs(train_state[:, j].astype(int))
    if len(on_durations) == 0:
        min_dwell_windows[app] = 1
        continue
    median_on_s = np.median(on_durations) * SAMPLE_INTERVAL_S
    # dwell expressed in units of STRIDE (the step between consecutive predicted windows)
    dwell = max(1, int(round((median_on_s / SAMPLE_INTERVAL_S) / STRIDE / 2)))
    min_dwell_windows[app] = dwell
    print(f"{app:16s} {median_on_s:13.0f} {dwell:19d}")


def apply_min_dwell_correction(binary_sequence, min_dwell):
    """Absorb any run (ON or OFF) shorter than min_dwell windows into its surrounding state."""
    seq = binary_sequence.copy()
    n = len(seq)
    i = 0
    while i < n:
        j = i
        while j < n and seq[j] == seq[i]:
            j += 1
        run_len = j - i
        if run_len < min_dwell and 0 < i and j < n and seq[i - 1] == seq[j % n]:
            seq[i:j] = seq[i - 1]
        i = j
    return seq


appliance         median_on(s)  min_dwell(windows)
laptop                     204                   1
washing_machine             12                   1
tv                          54                   1
fridge                      12                   1
microwave                   30                   1


## 15. Inference pipeline -- predict_timeline + explain_window

In [ ]:
@torch.no_grad()
def predict_timeline(model, df, kv_mean, kv_std, thresholds, min_dwell, window_size=WINDOW_SIZE, stride=STRIDE):
    model.eval()
    agg = df["aggregate"].values.astype(np.float32)
    time_s = df["time_s"].values
    starts = list(range(0, len(df) - window_size, stride))

    probs_all, kv_all, start_times, end_times = [], [], [], []
    for s in starts:
        e = s + window_size
        w = agg[s:e]
        kv_raw = compute_knowledge_vector(w)
        kv_norm = (kv_raw - kv_mean) / kv_std
        agg_norm = (w - w.mean()) / (w.std() + 1e-6)

        agg_t = torch.tensor(agg_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        kv_t = torch.tensor(kv_norm, dtype=torch.float32).unsqueeze(0).to(device)
        out = model(agg_t, kv_t)
        probs_all.append(torch.sigmoid(out["state_logits"])[0].cpu().numpy())
        kv_all.append(kv_raw)
        start_times.append(time_s[s]); end_times.append(time_s[e - 1])

    probs = np.stack(probs_all)
    state = np.zeros_like(probs, dtype=int)
    for j, app in enumerate(APPLIANCES):
        raw_state = (probs[:, j] >= thresholds[app]).astype(int)
        state[:, j] = apply_min_dwell_correction(raw_state, min_dwell[app])

    return {
        "start_time": np.array(start_times), "end_time": np.array(end_times),
        "probs": probs, "state": state, "kv": np.stack(kv_all),
    }


def explain_window(timeline, window_idx, appliance):
    j = APPLIANCES.index(appliance)
    prob = timeline["probs"][window_idx, j]
    state = timeline["state"][window_idx, j]
    kv = timeline["kv"][window_idx]
    kv_dict = dict(zip(KV_FEATURE_NAMES, kv))
    s, e = timeline["start_time"][window_idx], timeline["end_time"][window_idx]
    print(f"Window [{s:.0f}s - {e:.0f}s]: {appliance} predicted {'ON' if state else 'OFF'} "
          f"(confidence {prob:.2f}, dwell-corrected).")
    print(f"  Power evidence -> mean {kv_dict['mean_power']:.1f}W, max {kv_dict['max_power']:.1f}W, "
          f"duty_cycle {kv_dict['duty_cycle']:.2f}, spectral_entropy {kv_dict['spectral_entropy']:.2f}.")


## 16. Final evaluation on TEST set

In [ ]:
@torch.no_grad()
def get_test_probs(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        out = model(batch["aggregate"].to(device), batch["kv"].to(device))
        probs.append(torch.sigmoid(out["state_logits"]).cpu().numpy())
    return np.concatenate(probs)


TEST_PROBS = get_test_probs(model, test_loader)
test_labels = test_ds.labels

rows = []
for j, app in enumerate(APPLIANCES):
    y = test_labels[:, j].astype(int)
    prob = TEST_PROBS[:, j]
    pred_tuned = (prob >= BEST_THRESHOLDS[app]).astype(int)
    pred_half = (prob >= 0.5).astype(int)

    try:
        auc = roc_auc_score(y, prob)
    except ValueError:
        auc = np.nan

    rows.append({
        "Appliance": app,
        "Accuracy": accuracy_score(y, pred_tuned),
        "Precision": precision_score(y, pred_tuned, zero_division=0),
        "Recall": recall_score(y, pred_tuned, zero_division=0),
        "F1": f1_score(y, pred_tuned, zero_division=0),
        "F1_at_0.5": f1_score(y, pred_half, zero_division=0),
        "Threshold": BEST_THRESHOLDS[app],
        "ROC_AUC": auc,
    })

results = pd.DataFrame(rows)
display(results)

print()
print("Compare F1 (tuned) against F1_at_0.5 to see what threshold tuning bought. Compare each")
print("appliance's ROC_AUC against Section 6's diagnostic -- a weak F1 despite strong AUC points")
print("back to the shared-window dilution tradeoff (microwave, most likely), not a broken model.")


,Appliance,Accuracy,Precision,Recall,F1,F1_at_0.5,Threshold,ROC_AUC
0,laptop,0.806728,0.397518,0.521168,0.451022,0.468514,0.70,0.822527
1,washing_machine,0.983679,0.735880,0.939238,0.825215,0.805155,0.85,0.994986
2,tv,0.938336,0.632571,0.836423,0.720353,0.682593,0.80,0.972198
3,fridge,0.992635,0.718447,0.717672,0.718059,0.699601,0.60,0.974126
4,microwave,0.986946,0.402820,0.544959,0.463231,0.435257,0.20,0.981635



Compare F1 (tuned) against F1_at_0.5 to see what threshold tuning bought. Compare each
appliance's ROC_AUC against Section 6's diagnostic -- a weak F1 despite strong AUC points
back to the shared-window dilution tradeoff (microwave, most likely), not a broken model.


## 17. Sample explainability report

In [ ]:
one_week = df[df["time_s"] < 7 * 24 * 3600].reset_index(drop=True)
timeline = predict_timeline(model, one_week, train_ds.kv_mean, train_ds.kv_std, BEST_THRESHOLDS, min_dwell_windows)

for app in APPLIANCES:
    print(f"=== {app} ===")
    for idx in range(min(3, len(timeline["start_time"]))):
        explain_window(timeline, idx, app)
    print()


=== laptop ===
Window [0s - 688s]: laptop predicted ON (confidence 0.83, dwell-corrected).
  Power evidence -> mean 607.8W, max 904.0W, duty_cycle 1.00, spectral_entropy 0.88.
Window [153s - 872s]: laptop predicted ON (confidence 0.87, dwell-corrected).
  Power evidence -> mean 591.8W, max 904.0W, duty_cycle 1.00, spectral_entropy 0.87.
Window [282s - 1007s]: laptop predicted ON (confidence 0.78, dwell-corrected).
  Power evidence -> mean 559.5W, max 904.0W, duty_cycle 1.00, spectral_entropy 0.86.

=== washing_machine ===
Window [0s - 688s]: washing_machine predicted OFF (confidence 0.00, dwell-corrected).
  Power evidence -> mean 607.8W, max 904.0W, duty_cycle 1.00, spectral_entropy 0.88.
Window [153s - 872s]: washing_machine predicted OFF (confidence 0.01, dwell-corrected).
  Power evidence -> mean 591.8W, max 904.0W, duty_cycle 1.00, spectral_entropy 0.87.
Window [282s - 1007s]: washing_machine predicted OFF (confidence 0.32, dwell-corrected).
  Power evidence -> mean 559.5W, max 90

## 18. Save model + results

In [ ]:
RUN_NAME = datetime.datetime.now().strftime("nilm_joint_%Y%m%d_%H%M%S")

checkpoint = {
    "run_name": RUN_NAME,
    "timestamp": datetime.datetime.now().isoformat(),
    "model_state_dict": model.state_dict(),
    "best_config": best_config,
    "appliances": APPLIANCES,
    "kv_feature_names": KV_FEATURE_NAMES,
    "kv_mean": train_ds.kv_mean.tolist(),
    "kv_std": train_ds.kv_std.tolist(),
    "on_threshold": ON_THRESHOLD,
    "pos_weight_cap": POS_WEIGHT_CAP,
    "threshold_beta": THRESHOLD_BETA,
    "best_thresholds": BEST_THRESHOLDS,
    "min_dwell_windows": min_dwell_windows,
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "n_epochs_run": N_EPOCHS,
    "best_macro_f1_val": float(best_macro_f1),
    "test_results": results.to_dict(orient="records"),
}

torch.save(checkpoint, os.path.join(SAVE_DIR, f"{RUN_NAME}.pt"))

summary = {k: v for k, v in checkpoint.items() if k != "model_state_dict"}
with open(os.path.join(SAVE_DIR, f"{RUN_NAME}_summary.json"), "w") as f:
    _json.dump(summary, f, indent=2)

print(f"Saved -> {SAVE_DIR}/{RUN_NAME}.pt")
print(f"Saved -> {SAVE_DIR}/{RUN_NAME}_summary.json")


Saved -> /content/gdrive/MyDrive/NILM_JointModel/nilm_joint_20260920_123602.pt
Saved -> /content/gdrive/MyDrive/NILM_JointModel/nilm_joint_20260920_123602_summary.json
